Clean data into dataframe :)

In [9]:
import pandas as pd
from pathlib import Path
import numpy as np

In [22]:
# PARSING 
def parse_custom_csv(path):
    with open(path, "r") as f:
        lines = f.readlines()

    sections = {"META": [], "EVENTS": [], "METER": []}
    current = None

    for line in lines:
        line = line.strip()

        if line.startswith("#"):
            current = line.replace("#", "").strip() #define current section as the header
            continue

        if current and line:  # <-- FIX: skip empty lines
            sections[current].append(line)

    # META
    meta = {}
    for row in sections["META"]:
        k, v = row.split(",", 1)
        meta[k] = v

   # EVENTS
    header = sections["EVENTS"][0].split(",")
    data = [r.split(",") for r in sections["EVENTS"][1:]]
    events = pd.DataFrame(data, columns=header)

    # METER
    header = sections["METER"][0].split(",")
    data = [r.split(",") for r in sections["METER"][1:]]
    meter = pd.DataFrame(data, columns=header)

    return meta, events, meter

In [ ]:
# CLEANING
def clean_data(meta, events, meter, run_id):
    # Drop bad rows first
    events = events.dropna(subset=["payload_size", "tx_start", "rx_end"])

    # Convert types
    events["payload_size"] = events["payload_size"].astype(int)
    events["tx_start"] = pd.to_datetime(events["tx_start"])
    events["rx_end"] = pd.to_datetime(events["rx_end"])

    meter["timestamp"] = pd.to_datetime(meter["timestamp"])
    meter["v_shunt"] = meter["v_shunt"].astype(float)

    # Add run id
    events["run"] = run_id
    meter["run"] = run_id

    return meta, events, meter

In [6]:
# TIME ALIGNING
def align_time(events, meter):
    t0 = events["tx_start"].min()

    events["t_start_s"] = (events["tx_start"] - t0).dt.total_seconds()
    events["t_end_s"] = (events["rx_end"] - t0).dt.total_seconds()

    meter["t_s"] = (meter["timestamp"] - t0).dt.total_seconds()

    return events, meter

In [25]:
# LABELING PHASES
def label_phases(events, meter):
    meter["phase"] = "idle"
    meter["payload_size"] = 0

    for _, row in events.iterrows():
        mask = (
            (meter["t_s"] >= row["t_start_s"]) &
            (meter["t_s"] <= row["t_end_s"]) # make sure both start and end are in the transmission window
        )
        
        meter.loc[mask, "phase"] = "tx" # label phase as tx
        meter.loc[mask, "payload_size"] = int(row["payload_size"])

    return meter

Test the functions

In [26]:
file1 = Path("/Users/jude/Documents/GitHub/BTR/experiment/data/esp32/full_payload/esp32_full_payload_run01.csv")
meta1, events1, meter1 = parse_custom_csv(file1)
meta1, events1, meter1 = clean_data(meta1, events1, meter1, run_id=1)
events1, meter1 = align_time(events1, meter1)
meter1 = label_phases(events1, meter1)
print(events1.head())
print(meter1.head())


   run payload_size declared_size bytes_received                tx_start  \
0    1            1             1              1 2026-03-25 18:59:24.126   
1    1            2             2              2 2026-03-25 18:59:25.222   
2    1            4             4              4 2026-03-25 18:59:27.371   
3    1            8             8              8 2026-03-25 18:59:29.620   
4    1           16            16             16 2026-03-25 18:59:31.979   

                   rx_end complete verified  t_start_s  t_end_s  
0 2026-03-25 18:59:24.717     True     True      0.000    0.591  
1 2026-03-25 18:59:26.866     True     True      1.096    2.740  
2 2026-03-25 18:59:29.119     True     True      3.245    4.993  
3 2026-03-25 18:59:31.473     True     True      5.494    7.347  
4 2026-03-25 18:59:33.624     True     True      7.853    9.498  
                timestamp  v_shunt  run    t_s phase  payload_size
0 2026-03-25 18:59:24.130  0.10431    1  0.004    tx             1
1 2026-03-25 

Run on all files

In [27]:
all_meter = []
all_events = []

data_dir = Path("/Users/jude/Documents/GitHub/BTR/experiment/data/esp32/full_payload")

for i, file in enumerate(sorted(data_dir.glob("*.csv"))):
    meta, events, meter = parse_custom_csv(file)
    meta, events, meter = clean_data(meta, events, meter, run_id=i)

    events, meter = align_time(events, meter)
    meter = label_phases(events, meter)

    all_meter.append(meter)
    all_events.append(events)

meter_df = pd.concat(all_meter, ignore_index=True)
events_df = pd.concat(all_events, ignore_index=True)

Plot some stuff